# Lab 5.5 &mdash; Challenge: The Scorecard

**Level:** Advanced &middot; challenge &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 5 &mdash; Multi-Agent Collaboration &amp; Orchestration**

### What you'll do
- State the ground truth &mdash; what the right recommendation actually is, and why
- Run a single agent and a four-specialist graph over the same cases
- Price both: quality, tokens, and the critical path
- Turn it into a decision that names what a wrong answer costs

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The module's deliverable.** Not a graph &mdash; an argument about whether to build one,
> with numbers in it that a risk owner can agree or disagree with.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-5-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 5 labs -- the same payment exceptions, now worked
# by several agents at once, and finally priced against the single agent from Day 1.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the specialists
# Deterministic stand-ins. Each takes the run state and returns a PARTIAL state -- exactly
# the LangGraph node shape from Module 3 -- and reports what it spent. No model is called,
# so a graph's structure AND its cost can be graded offline and exactly. The "Run it for
# real" cells put the sandbox model behind the same interface.

SANCTIONS_WATCH = {"NORTHWIND"}

COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}


def agent_ledger(state: dict) -> dict:
    """Read the payment named in the state."""
    ref = state.get("ref")
    record = LEDGER.get(ref)
    if record is None:
        return {"problems": [f"no payment on file with reference {ref!r}"],
                "tokens": COST["ledger"]}
    return {"facts": {"ref": ref, **record},
            "findings": [{"by": "ledger", "source": "ledger",
                          "claim": f"{ref} is {record['status']} "
                                   f"for {record['amount']:,.2f} {record['ccy']}"}],
            "tokens": COST["ledger"]}


def agent_policy(state: dict) -> dict:
    """Say what the operating policy is for whatever went wrong."""
    code = (state.get("facts") or {}).get("reason_code")
    if code is None:
        return {"problems": ["policy ran before the reason code existed"],
                "tokens": COST["policy"]}
    return {"findings": [{"by": "policy", "source": "policy",
                          "claim": POLICY.get(code, f"no policy on file for {code}")}],
            "needs_human": code in NEEDS_HUMAN,
            "tokens": COST["policy"]}


def agent_sanctions(state: dict) -> dict:
    """A set-membership test. No model needed, and none used -- note the cost column."""
    counterparty = (state.get("facts") or {}).get("counterparty")
    listed = counterparty in SANCTIONS_WATCH
    return {"findings": [{"by": "sanctions", "source": "watchlist",
                          "claim": f"{counterparty} is "
                                   f"{'ON the watchlist' if listed else 'not on the watchlist'}"}],
            "blocked": listed,
            "tokens": COST["sanctions"]}


def agent_writer(state: dict) -> dict:
    """Turn whatever findings arrived into one recommendation."""
    findings = state.get("findings") or []
    if (state.get("facts") or {}).get("status") == "settled":
        action = "no action"                      # nothing to release; it already went
    elif state.get("blocked") or state.get("needs_human"):
        action = "hold for a human"
    else:
        action = "release"
    return {"recommendation": action,
            "rationale": [f["claim"] for f in findings],
            "tokens": COST["writer"]}


AGENTS = {"ledger": agent_ledger, "policy": agent_policy,
          "sanctions": agent_sanctions, "writer": agent_writer}
print("specialists:", ", ".join(AGENTS))

## Concept

You have a single agent from Day 1 and a graph from this module. The question is not which is
more sophisticated. It is whether the errors the graph prevents are worth the tokens it burns.

**A note on what is graded here.** The two implementations below are readable *models* of the two
designs &mdash; deterministic, so the comparison is exact and repeatable offline. What is graded is
the **scorecard machinery**: the ground truth, the metric, the cost model and the decision rule.
Point them at your own system and the machinery is unchanged; only the numbers move.

## Section 1 &mdash; Ground truth

Before measuring anything, say what the right answer is. Two extra cases are added here: a
watchlisted counterparty whose reason code says nothing about sanctions. Those are the cases that
separate the two designs, and real case files are full of them.

In [ ]:
EXTRA = {
    "PMT-1006": {"amount": 62000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "failed", "value_date": "2026-09-03",
                 "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1007": {"amount":  8400.00, "ccy": "GBP", "counterparty": "NORTHWIND",
                 "status": "failed", "value_date": "2026-09-03",
                 "reason_code": "INVALID_IBAN"},
}
CASES = {**LEDGER, **EXTRA}


def expected_recommendation(record: dict) -> str:
    """The correct answer for one payment, independent of any agent.

    Three outcomes: a settled payment needs nothing; anything a human must decide, or any
    watchlisted counterparty, is held; everything else proceeds.
    """
    if record["status"] == "settled":
        return "no action"
    if record["reason_code"] in NEEDS_HUMAN or record["counterparty"] in SANCTIONS_WATCH:
        return "hold for a human"
    return "proceed"


def eval_cases() -> list:
    """The cases paired with their ground-truth answers. Built on demand, not at import:
    a module-level call into a function with a blank in it crashes the whole cell."""
    return [(ref, expected_recommendation(rec)) for ref, rec in sorted(CASES.items())]

In [ ]:
# --- Self-check: Section 1
check("a settled payment needs no action",
      lambda: expected_recommendation(CASES["PMT-1001"]) == "no action")
check("a sanctions review is held",
      lambda: expected_recommendation(CASES["PMT-1005"]) == "hold for a human")
check("so is a limit breach",
      lambda: expected_recommendation(CASES["PMT-1003"]) == "hold for a human")
check("an ordinary funding failure proceeds",
      lambda: expected_recommendation(CASES["PMT-1002"]) == "proceed")
check("a watchlisted counterparty is held even when its reason code is mundane",
      lambda: expected_recommendation(CASES["PMT-1006"]) == "hold for a human",
      "nothing in INSUFFICIENT_FUNDS hints at sanctions -- the counterparty is the whole reason")
check("and again for the second one",
      lambda: expected_recommendation(CASES["PMT-1007"]) == "hold for a human")
check("the eval set covers all three outcomes",
      lambda: {e for _, e in eval_cases()} == {"no action", "hold for a human", "proceed"})
check("seven cases in total",
      lambda: len(eval_cases()) == 7)

## Section 2 &mdash; The two designs

The single agent reads the payment, consults policy when there is a reason code, and screens the
counterparty **only when something in the case points at sanctions**. That is not a strawman: it
is what one prompt with a step budget does &mdash; it follows the happy path the case suggests.

The graph runs every specialist every time. That is the whole of its advantage, and the whole of
its cost.

In [ ]:
def single_agent(ref: str) -> dict:
    """One agent, one context. Reads, consults policy, screens only if prompted to."""
    record = CASES.get(ref)
    used = ["ledger"]
    if record is None:
        return {"recommendation": "no action", "used": used}
    needs_human = False
    if record["reason_code"]:
        used.append("policy")
        needs_human = record["reason_code"] in NEEDS_HUMAN
    # it screens only when the case points that way -- nothing else prompts it to
    if record["reason_code"] == "SANCTIONS_REVIEW":
        used.append("sanctions")
        needs_human = needs_human or record["counterparty"] in SANCTIONS_WATCH
    used.append("writer")
    if record["status"] == "settled":
        action = "no action"
    elif needs_human:
        action = "hold for a human"
    else:
        action = "proceed"
    return {"recommendation": action, "used": used}


def graph_agent(ref: str) -> dict:
    """Supervisor plus four specialists. Every specialist runs, every time."""
    record = CASES.get(ref)
    used = ["ledger", "policy", "sanctions", "writer"]
    if record is None:
        return {"recommendation": "no action", "used": used, "dispatches": 4}
    blocked = record["counterparty"] in SANCTIONS_WATCH
    needs_human = record["reason_code"] in NEEDS_HUMAN
    if record["status"] == "settled":
        action = "no action"
    elif blocked or needs_human:
        action = "hold for a human"
    else:
        action = "proceed"
    return {"recommendation": action, "used": used, "dispatches": 4}


def _side_by_side():
    print(f"  {'case':10}{'expected':18}{'single':18}{'graph':18}")
    print("  " + "-" * 62)
    for ref, expected in eval_cases():
        s, g = single_agent(ref)["recommendation"], graph_agent(ref)["recommendation"]
        flag = "" if s == expected else "   <-- single is wrong"
        print(f"  {ref:10}{expected:18}{s:18}{g:18}{flag}")
guard(_side_by_side)

## Section 3 &mdash; Score and price them

Quality is agreement with the ground truth. Cost is what each design woke up. The supervisor is
charged once per dispatch, because routing is not free.

In [ ]:
COST = {"supervisor": 120, "ledger": 380, "policy": 420, "sanctions": 90, "writer": 610}
CONTEXT_TOKENS = 600      # the case file: payment, policy text, findings so far

def run_cost(result: dict) -> int:
    """Tokens for one case.

    A single agent holds ONE context and reuses it across its own turns. A graph re-sends the
    case context to every specialist it dispatches, and pays the supervisor to route each time.
    That re-sending is where the cost multiple comes from -- not from the agents themselves.
    """
    specialists = sum(COST[a] for a in result["used"])
    dispatches = result.get("dispatches", 0)
    contexts = dispatches if dispatches else 1
    return specialists + CONTEXT_TOKENS * contexts + COST["supervisor"] * dispatches


def evaluate(design) -> dict:
    """Run one design over every case. Returns correct count, accuracy and total tokens."""
    correct, tokens, misses = 0, 0, []
    for ref, expected in eval_cases():
        result = design(ref)
        tokens += run_cost(result)
        if result["recommendation"] == expected:
            correct += 1
        else:
            misses.append((ref, expected, result["recommendation"]))
    return {"correct": correct, "accuracy": correct / len(eval_cases()),
            "tokens": tokens, "misses": misses}


def critical_path(design_name: str) -> int:
    """Waves on the critical path -- what parallelism can and cannot shorten."""
    return 2 if design_name == "single" else 3

In [ ]:
# --- Self-check: Section 3
def S():
    return evaluate(single_agent)
def G():
    return evaluate(graph_agent)

check("the graph gets every case right",
      lambda: G()["correct"] == len(eval_cases()))
check("the single agent does not",
      lambda: S()["correct"] < len(eval_cases()))
check("and it misses exactly the two watchlisted-but-mundane cases",
      lambda: sorted(r for r, _, _ in S()["misses"]) == ["PMT-1006", "PMT-1007"],
      "the cases where nothing in the reason code told it to look")
check("both of its misses are the dangerous direction -- proceeding when it should hold",
      lambda: all(got == "proceed" and want == "hold for a human"
                  for _, want, got in S()["misses"]))
check("the graph costs more",
      lambda: G()["tokens"] > S()["tokens"])
check("and by a multiple worth naming, not a rounding error",
      lambda: G()["tokens"] / S()["tokens"] > 1.5)
check("the graph's critical path is longer too",
      lambda: critical_path("graph") > critical_path("single"))

def _score():
    s, g = S(), G()
    print(f"  {'':16}{'single':>12}{'graph':>12}")
    print("  " + "-" * 40)
    print(f"  {'accuracy':16}{s['accuracy']:>11.0%}{g['accuracy']:>12.0%}")
    print(f"  {'tokens':16}{s['tokens']:>12}{g['tokens']:>12}")
    print(f"  {'cost multiple':16}{'1.0x':>12}{g['tokens'] / s['tokens']:>11.1f}x")
    print(f"  {'critical path':16}{critical_path('single'):>12}{critical_path('graph'):>12}")
guard(_score)

## Section 4 &mdash; The decision

Two numbers finish the argument, and neither is technical: what one wrong recommendation costs to
put right, and what a token costs. Put your own figures in.

In [ ]:
TOKEN_PRICE = 0.0000006      # currency per token -- substitute your own
ERROR_COST  = 2500.0         # what putting one wrong recommendation right costs you

def verdict(single: dict, graph: dict,
            error_cost: float = ERROR_COST, token_price: float = TOKEN_PRICE) -> dict:
    """Ship the graph only if the errors it prevents are worth more than the tokens it burns."""
    errors_prevented = graph["correct"] - single["correct"]
    value_saved = errors_prevented * error_cost
    extra_spend = (graph["tokens"] - single["tokens"]) * token_price
    return {"errors_prevented": errors_prevented,
            "value_saved": round(value_saved, 4),
            "extra_spend": round(extra_spend, 4),
            "decision": "ship the graph" if value_saved > extra_spend else "don't"}


def breakeven_error_cost(single: dict, graph: dict,
                         token_price: float = TOKEN_PRICE) -> float:
    """How expensive one error has to be before the graph pays for itself."""
    prevented = graph["correct"] - single["correct"]
    if prevented <= 0:
        return float("inf")
    return (graph["tokens"] - single["tokens"]) * token_price / prevented

In [ ]:
# --- Self-check: Section 4
check("the graph prevents two errors on this eval set",
      lambda: verdict(S(), G())["errors_prevented"] == 2)
check("at a realistic error cost, ship it",
      lambda: verdict(S(), G())["decision"] == "ship the graph")
check("if an error costs almost nothing, do not",
      lambda: verdict(S(), G(), error_cost=0.0001)["decision"] == "don't",
      "the same graph, the same quality gain, the opposite answer -- the economics decide")
check("the breakeven is a number you can quote",
      lambda: 0 < breakeven_error_cost(S(), G()) < ERROR_COST)
check("and the decision flips either side of it",
      lambda: verdict(S(), G(), error_cost=breakeven_error_cost(S(), G()) * 2)["decision"]
              == "ship the graph"
          and verdict(S(), G(), error_cost=breakeven_error_cost(S(), G()) / 2)["decision"]
              == "don't")
check("a graph that prevents nothing never pays, at any error cost",
      lambda: breakeven_error_cost(G(), G()) == float("inf"),
      "two designs of equal quality are decided on cost alone, and the cheaper one wins")

def _verdict():
    v = verdict(S(), G())
    print(f"  errors prevented per {len(eval_cases())} cases : {v['errors_prevented']}")
    print(f"  value saved                     : {v['value_saved']}")
    print(f"  extra spend                     : {v['extra_spend']}")
    print(f"  breakeven cost of one error     : {breakeven_error_cost(S(), G()):.4f}")
    print()
    print(f"  DECISION: {v['decision']}")
    print()
    print("  Read it as: the graph pays for itself as long as one wrong recommendation")
    print(f"  costs more than {breakeven_error_cost(S(), G()):.4f} to put right.")
guard(_verdict)

## Run it for real

Everything above is deterministic, which is what makes it repeatable. Now run the same seven cases
through the model twice and see how stable the answer is &mdash; because a quality number from a
single run of a non-deterministic system is an anecdote.

In [ ]:
if llm_ready():
    def _stability():
        def model_recommendation(ref):
            rec = CASES[ref]
            reply = ask("You are a payments operations agent. Reply with exactly one of: "
                        "no action / hold for a human / proceed.\n\n"
                        f"Payment: {json.dumps(rec)}\n"
                        f"Reference: {ref}\n"
                        f"Watchlisted counterparties: {sorted(SANCTIONS_WATCH)}\n"
                        f"Reason codes that require a human: {sorted(NEEDS_HUMAN)}",
                        system="Reply with the phrase alone.")
            return (reply or "").strip().lower().rstrip(".")
        for run in (1, 2):
            correct = sum(1 for ref, expected in eval_cases()
                          if model_recommendation(ref) == expected)
            print(f"  run {run}: {correct}/{len(eval_cases())} correct")
    guard(_stability)

### Read it

If the two runs disagree, you have just met Module 7's opening problem: the same input, twice, two
different answers. That does not invalidate the scorecard &mdash; it tells you the scorecard needs
repeats and a confidence interval, which is Day 3's work.

**What you take from Module 5:** a supervisor is a router you can measure; a handoff carries only
what you put in it; parallel branches lose findings unless you declare a reducer; disagreement is
settled by authority and provenance, not by counting; the human is a node with a deadline and an
escalation ladder; and the graph earns its place with a number or it does not earn it at all.

In [ ]:
score()

## Your turn

1. Route by value: run the graph only above a threshold and the single agent below it. Find the
   threshold that maximises value, and check whether it is one you would defend to a regulator.
2. `single_agent` misses the two cases nothing prompted it to look at. Fix it with one sentence of
   prompt &mdash; &ldquo;always screen the counterparty&rdquo; &mdash; and re-score. If that closes the gap, the
   honest answer for this workload is that you never needed the graph.
3. The scorecard has no row for operating cost: five agents are five things to trace, alert on and
   page someone about. Add that row in whatever unit you can defend, and see whether the decision
   survives it.